# Prvi probni kolokvij

## 1. Zadatak (50)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sbn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier

### 1.1 Rad s podacima – obvezni dio

In [ ]:
# Učitavanje podataka
data = pd.read_csv('os.csv')
print("Prvih 5 redaka:")
print(data.head())

In [ ]:
# Osnovna analiza
print("Oblik skupa podataka:", data.shape)
print("\nInformacije o stupcima:")
print(data.info())
print("\nStatistički opis:")
print(data.describe())

In [ ]:
# Pregled nedostajućih vrijednosti po značajkama
print("Broj NaN vrijednosti po stupcima:")
print(data.isnull().sum())

In [ ]:
# Uklanjamo stupce koji NEMAJU vrijednosti za svih 365 dana (tj. imaju barem jedan NaN)
cols_before = set(data.columns)
data_clean = data.dropna(axis=1)
cols_after = set(data_clean.columns)
print("Uklonjeni stupci:", cols_before - cols_after)

# Uklanjamo stupac 'date'
data_clean = data_clean.drop(columns=['date'])
print("\nPreostale značajke:", list(data_clean.columns))
print("\nPrvih 5 redaka očišćenog skupa:")
print(data_clean.head())

### 1.2 Rad s podacima – dodatni dio

In [ ]:
# Ponovo učitavamo originalne podatke kako bismo zadržali sve stupce za dodatni dio
data_extra = pd.read_csv('os.csv')

# Pronalazimo stupac koji ima samo nekoliko NaN vrijednosti
print("Broj NaN po stupcima:")
print(data_extra.isnull().sum())

# 'prcp' (oborine) ima mali broj NaN vrijednosti – zamijenimo ih s 0
nan_idx = data_extra[data_extra['prcp'].isnull()].index
print("\nIndeksi gdje prcp ima NaN:", list(nan_idx))

data_extra['prcp'] = data_extra['prcp'].fillna(0)
print("NaN vrijednosti u prcp nakon popunjavanja:", data_extra['prcp'].isnull().sum())

In [ ]:
# Uključivanje datuma u model – koristimo dan u godini (1–365)
# To zadržava redoslijed i sezonalnost bez uvođenja kategoričkih varijabli
data_extra['date'] = pd.to_datetime(data_extra['date'])
data_extra['day_of_year'] = data_extra['date'].dt.dayofyear

print("Primjer day_of_year:")
print(data_extra[['date', 'day_of_year']].head(10))

In [ ]:
# Dodavanje značajke godišnjeg doba
# proljeće: 21.3. (dan 80), ljeto: 21.6. (dan 172), jesen: 21.9. (dan 264), zima: 21.12. (dan 355)

def get_season(doy):
    if doy < 80 or doy >= 355:
        return 0  # zima
    elif doy < 172:
        return 1  # proljeće
    elif doy < 264:
        return 2  # ljeto
    else:
        return 3  # jesen

data_extra['season'] = data_extra['day_of_year'].apply(get_season)

season_names = {0: 'zima', 1: 'proljeće', 2: 'ljeto', 3: 'jesen'}
print("Raspodjela godišnjih doba:")
print(data_extra['season'].map(season_names).value_counts())

# Uklanjamo stupce s previše NaN (snow, wdir, tsun) i date
cols_to_drop = ['snow', 'wdir', 'tsun', 'date']
data_extra = data_extra.drop(columns=cols_to_drop)
print("\nKonačne značajke u data_extra:", list(data_extra.columns))

### 1.3 Vizualizacija značajki kroz vrijeme

In [ ]:
# Koristimo data_clean (obvezni dio) – bez date stupca, samo numeričke značajke
# Trebamo index za os x (dani 1–365)
features_to_plot = list(data_clean.columns)
days = range(1, len(data_clean) + 1)

fig, axes = plt.subplots(nrows=len(features_to_plot), ncols=1, figsize=(14, 3 * len(features_to_plot)))

for i, col in enumerate(features_to_plot):
    axes[i].plot(days, data_clean[col], label=col, color='steelblue', linewidth=1.2)
    axes[i].set_title(f'{col} kroz 2025. godinu')
    axes[i].set_xlabel('Dan u godini')
    axes[i].set_ylabel(col)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 1.4 Odabir značajki i treniranje modela

In [ ]:
# X = svi dani od prvog do predzadnjeg
# y = tavg za dane od drugog do zadnjeg
# Ideja: vrijednosti od jučer predviđaju temperaturu danas

X = data_clean.iloc[:-1].values       # predzadnji dan je zadnji u X
y = data_clean['tavg'].iloc[1:].values  # od drugog dana nadalje

print("Oblik X:", X.shape)
print("Oblik y:", y.shape)

In [ ]:
# Razdvajanje 80:20 – shuffle=False jer su vremenski podaci, red je bitan!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("Trening skup:", X_train.shape)
print("Test skup:", X_test.shape)

In [ ]:
# Treniranje linearnog regresijskog modela
model = LinearRegression()
model.fit(X_train, y_train)

print("Model uspješno natreniran.")

### 1.5 Ocjene modela

In [ ]:
# Predikcija na testnom skupu
y_pred = model.predict(X_test)

# Metrike
r2  = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R² score:  {r2:.4f}")
print(f"MSE:       {mse:.4f}")
print(f"MAE:       {mae:.4f}")

In [ ]:
# Parametri modela – koeficijenti za svaku značajku
feature_names = list(data_clean.columns)
coefs = pd.Series(model.coef_, index=feature_names)

print("Intercept:", round(model.intercept_, 4))
print("\nKoeficijenti po značajkama:")
print(coefs.round(4).to_string())

most_important = coefs.abs().idxmax()
print(f"\nZnačajka koja najviše utječe na predviđanje temperature: '{most_important}' "
      f"(koeficijent: {coefs[most_important]:.4f})")
print("\nKomentar: 'tavg' (prosječna temperatura prethodnog dana) ima najveći koeficijent,"
      " što je i intuitivno – temperatura je autokorelirana, tj. dan s visokom temperaturom"
      " najčešće slijedi drugi topli dan.")

In [ ]:
# MAE – prosječno apsolutno odstupanje od stvarne temperature
print(f"MAE = {mae:.4f} °C")
print("\nKomentar: MAE nam govori da model u prosjeku griješi za otprilike",
      round(mae, 2), "°C pri predviđanju dnevne temperature.")

In [ ]:
# Graf: stvarne vrijednosti (zelena) vs predviđanje (plava isprekidana)
plt.figure(figsize=(14, 5))
plt.plot(y_test, color='green', label='Stvarne vrijednosti', linewidth=1.5)
plt.plot(y_pred, color='blue', linestyle='--', label='Predviđanje modela', linewidth=1.5)
plt.title('Predviđanje prosječne dnevne temperature (testni skup)')
plt.xlabel('Dan (testni skup)')
plt.ylabel('Temperatura (°C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# 2. Zadatak (50)

### 2.1 Rad s podacima – obvezni dio

In [ ]:
# Učitavanje Iris skupa podataka
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target

print("Prvih 5 redaka:")
print(df.head())

In [ ]:
print("Oblik:", df.shape)
print("\nStruktura:")
print(df.info())
print("\nStatistički opis:")
print(df.describe())

In [ ]:
print("Nazivi klasa:", iris.target_names)
print("\nBroj primjera po klasi:")
print(df['target'].value_counts().sort_index())
print("\nOpis klasa:")
for i, name in enumerate(iris.target_names):
    count = (df['target'] == i).sum()
    print(f"  Klasa {i} ({name}): {count} primjera")

### 2.2 Vizualizacija – dodatni dio

In [ ]:
colors = ['red', 'green', 'blue']
labels = iris.target_names

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Par 1: petal length vs petal width
for cls in range(3):
    mask = df['target'] == cls
    axes[0].scatter(df.loc[mask, 'petal length (cm)'],
                    df.loc[mask, 'petal width (cm)'],
                    c=colors[cls], label=labels[cls], alpha=0.7)
axes[0].set_xlabel('Petal length (cm)')
axes[0].set_ylabel('Petal width (cm)')
axes[0].set_title('Petal length vs Petal width')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Par 2: sepal length vs sepal width
for cls in range(3):
    mask = df['target'] == cls
    axes[1].scatter(df.loc[mask, 'sepal length (cm)'],
                    df.loc[mask, 'sepal width (cm)'],
                    c=colors[cls], label=labels[cls], alpha=0.7)
axes[1].set_xlabel('Sepal length (cm)')
axes[1].set_ylabel('Sepal width (cm)')
axes[1].set_title('Sepal length vs Sepal width')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKomentar:")
print("- Na grafu petal length vs petal width jasno se vidi linearna separabilnost:"
      " klasa 0 (setosa) je potpuno odvojena od preostale dvije.")
print("- Klase 1 i 2 se donekle preklapaju, ali su uglavnom odvojive linearnom granicom.")
print("- Na grafu sepal length vs sepal width preklapanje klasa 1 i 2 je veće,"
      " pa taj par nije idealan za linearnu separaciju.")

### 2.3 Predprocesiranje i osnovna klasifikacija

In [ ]:
X_iris = df.drop(columns='target').values
y_iris = df['target'].values

# Razdvajanje 80:20
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

# Standardizacija
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

print("Trening skup:", X_tr_s.shape)
print("Test skup:",    X_te_s.shape)

In [ ]:
# Linearni klasifikator – SVC s linearnom jezgrom
svc_linear = SVC(kernel='linear', random_state=42)
svc_linear.fit(X_tr_s, y_tr)

y_pred_lin = svc_linear.predict(X_te_s)
acc_linear = accuracy_score(y_te, y_pred_lin)
print(f"Točnost linearnog SVC na testnom skupu: {acc_linear:.4f} ({acc_linear*100:.1f}%)")

In [ ]:
# Matrica zabune
cm = confusion_matrix(y_te, y_pred_lin)
print("Matrica zabune:")
print(cm)

plt.figure(figsize=(6, 4))
sbn.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title('Matrica zabune – linearni SVC')
plt.ylabel('Stvarna klasa')
plt.xlabel('Predviđena klasa')
plt.tight_layout()
plt.show()

# Ručni izračun preciznosti i odziva za svaku klasu
print("\nRučni izračun preciznosti i odziva:")
for i, name in enumerate(iris.target_names):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP   # ostale klase predviđene kao i
    FN = cm[i, :].sum() - TP   # klasa i predviđena kao nešto drugo
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    print(f"  {name:15s} → Preciznost: {precision:.4f}  |  Odziv: {recall:.4f}")

### 2.4 Vlastita implementacija One-vs-One klasifikatora

In [ ]:
# Treniranje – za svaki par klasa treniramo jedan binarni SVC
classes = np.unique(y_tr)
classifiers = {}  # ključ: (klasa_a, klasa_b), vrijednost: natrenirani model

for i in range(len(classes)):
    for j in range(i + 1, len(classes)):
        cls_a, cls_b = classes[i], classes[j]
        
        # Uzimamo samo primjere koji pripadaju klasi a ili b
        mask = (y_tr == cls_a) | (y_tr == cls_b)
        X_pair = X_tr_s[mask]
        y_pair = y_tr[mask]
        
        clf = SVC(kernel='linear', random_state=42)
        clf.fit(X_pair, y_pair)
        classifiers[(cls_a, cls_b)] = clf

print(f"Broj natrenianih klasifikatora: {len(classifiers)}")
print("Parovi:", list(classifiers.keys()))

In [ ]:
# Predikcija glasanjem
def ovo_predict(X, classifiers, classes):
    votes = np.zeros((len(X), len(classes)))
    
    for (cls_a, cls_b), clf in classifiers.items():
        preds = clf.predict(X)
        for idx, pred in enumerate(preds):
            votes[idx, pred] += 1  # pobjednička klasa dobiva glas
    
    return classes[np.argmax(votes, axis=1)]

y_pred_ovo = ovo_predict(X_te_s, classifiers, classes)
acc_ovo = accuracy_score(y_te, y_pred_ovo)

print(f"Točnost vlastite OvO implementacije: {acc_ovo:.4f} ({acc_ovo*100:.1f}%)")

cm_ovo = confusion_matrix(y_te, y_pred_ovo)
print("\nMatrica zabune:")
print(cm_ovo)

plt.figure(figsize=(6, 4))
sbn.heatmap(cm_ovo, annot=True, fmt='d', cmap='Greens',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title('Matrica zabune – vlastiti OvO')
plt.ylabel('Stvarna klasa')
plt.xlabel('Predviđena klasa')
plt.tight_layout()
plt.show()

### 2.5 Gotovi sklearn klasifikatori

In [ ]:
# OneVsOneClassifier
ovo_clf = OneVsOneClassifier(SVC(kernel='linear', random_state=42))
ovo_clf.fit(X_tr_s, y_tr)
y_pred_sklearn_ovo = ovo_clf.predict(X_te_s)
acc_sklearn_ovo = accuracy_score(y_te, y_pred_sklearn_ovo)
print(f"Točnost sklearn OvO:  {acc_sklearn_ovo:.4f} ({acc_sklearn_ovo*100:.1f}%)")

# OneVsRestClassifier
ovr_clf = OneVsRestClassifier(SVC(kernel='linear', random_state=42))
ovr_clf.fit(X_tr_s, y_tr)
y_pred_sklearn_ovr = ovr_clf.predict(X_te_s)
acc_sklearn_ovr = accuracy_score(y_te, y_pred_sklearn_ovr)
print(f"Točnost sklearn OvR:  {acc_sklearn_ovr:.4f} ({acc_sklearn_ovr*100:.1f}%)")

print("\nUsporedba svih pristupa:")
print(f"  Linearni SVC (direktno):  {acc_linear:.4f}")
print(f"  Vlastiti OvO:             {acc_ovo:.4f}")
print(f"  sklearn OvO:              {acc_sklearn_ovo:.4f}")
print(f"  sklearn OvR:              {acc_sklearn_ovr:.4f}")

In [ ]:
%%timeit
ovo_t = OneVsOneClassifier(SVC(kernel='linear', random_state=42))
ovo_t.fit(X_tr_s, y_tr)

In [ ]:
%%timeit
ovr_t = OneVsRestClassifier(SVC(kernel='linear', random_state=42))
ovr_t.fit(X_tr_s, y_tr)

In [ ]:
print("Komentar o brzini:")
print("OvO trenira k*(k-1)/2 = 3 klasifikatora na manjim podskupovima podataka.")
print("OvR trenira k = 3 klasifikatora na cijelom skupu.")
print("Na malom skupu kao što je Iris razlike su minimalne, ali na većim skupovima")
print("OvO može biti sporiji zbog većeg broja klasifikatora, unatoč manjim podskupovima.")